In [1]:
import tempfile
from unittest import mock
from absl.testing import absltest
from absl.testing import parameterized
import pandas as pd
import numpy as np
import xarray as xr
import tensorflow as tf

# from meridian.planner import roi_to_coefficients_converter
from meridian.data import input_data
from meridian.model import model
from meridian.model import spec
from meridian import constants


In [3]:
class ROIToCoefficientsConverterTest(parameterized.TestCase):

  def setUp(self):
    """Set up test fixtures."""
    super().setUp()

    # Sample model configuration
    self.model_config = {
      'time_col': 'week',
      'geo_col': 'geo',
      'population_col': 'population',
      'kpi_type': 'non_revenue',
      'kpi_col': 'conversions',
      'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
      'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
      'media_channels': ['Channel0', 'Channel1', 'Channel2'],
      'reach_cols': ['Channel3_reach'],
      'frequency_cols': ['Channel3_frequency'],
      'rf_spend_cols': ['Channel3_spend'],
      'rf_channels': ['Channel3']
    }

    # Sample ROI data
    self.sample_roi_data = pd.DataFrame({
      'geo': ['Geo0', 'Geo1', 'Geo2'],
      'Channel0': [2.5, 3.2, 2.8],  # ROI values
      'Channel1': [1.8, 2.1, 1.9],
      'Channel2': [3.1, 2.7, 3.4],
      'Channel3': [4.2, 3.8, 4.5]   # RF channel ROI
    })

    # Sample parameters data
    self.sample_parameters = pd.DataFrame({
      'MediaVariable': ['Channel0', 'Channel1', 'Channel2', 'Channel3'],
      'Adstock': [0.51, 0.29, 0.17, 0.6],
      'Inflexion': [1.53, 1.23, 1.16, 1.4],
      'Slope': [1.0, 1.0, 1.0, 3.0]
    })

    # Create mock InputData object
    self.mock_input_data = self._create_mock_input_data()

  def _create_mock_input_data(self):
    """Create a mock InputData object for testing."""
    # Create mock data with proper dimensions
    n_geos, n_times, n_media_channels = 3, 52, 3  # 3 geos, 52 weeks, 3 media channels
    n_rf_channels = 1

    # Create mock arrays
    mock_input_data = mock.MagicMock(spec=input_data.InputData)

    # Set up geo coordinates
    mock_input_data.geo = xr.DataArray(['Geo0', 'Geo1', 'Geo2'], dims='geo')

    # Set up population data
    mock_input_data.population = xr.DataArray([10000, 15000, 12000], dims='geo')

    # Set up revenue per kpi (optional)
    mock_input_data.revenue_per_kpi = xr.DataArray(
        np.ones((n_geos, n_times)),
        dims=['geo', 'time'],
        coords={'geo': ['Geo0', 'Geo1', 'Geo2'], 'time': range(n_times)}
    )

    # Set up media spend data
    media_spend_data = np.random.uniform(1000, 5000, (n_geos, n_times, n_media_channels))
    mock_input_data.media_spend = xr.DataArray(
        media_spend_data,
        dims=['geo', 'time', 'media_channel'],
        coords={
            'geo': ['Geo0', 'Geo1', 'Geo2'],
            'time': range(n_times),
            'media_channel': ['Channel0', 'Channel1', 'Channel2']
        }
    )

    # Set up RF spend data
    rf_spend_data = np.random.uniform(2000, 8000, (n_geos, n_times, n_rf_channels))
    mock_input_data.rf_spend = xr.DataArray(
        rf_spend_data,
        dims=['geo', 'time', 'rf_channel'],
        coords={
            'geo': ['Geo0', 'Geo1', 'Geo2'],
            'time': range(n_times),
            'rf_channel': ['Channel3']
        }
    )

    return mock_input_data

In [4]:
mock_input_data.media_spend

NameError: name 'mock_input_data' is not defined